# Import Packages/Libraries

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.stats import mode
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load Data

In [2]:
train = pd.read_csv("../data/training.csv")
task = pd.read_csv("../data/task.csv")

features = [col for col in train.columns if col not in ("i", "y")]   # 'i' é só o número do registo
X = train[features].values
y = train["y"].values
class_names = ["sem desconto", "com desconto"]

print("Sample:")
print(X[:5])
print("\nSample Label:")
print(y[:5])
print("\nClass Names:")
print(class_names)
print("\nCharacteristics Names:")
print(features)
print("\nValores em falta por variável:")
print(train[features].isna().sum()[lambda s: s > 0].to_dict())
print(f"\nProporção de produtos com desconto: {y.mean():.3f}")

Sample:
[[ 0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    4.    2.    0.5   1.    1.69 15.    4.
   4.   37.  ]
 [ 0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    1.    0.    1.    3.    1.    1.    6.49  4.    2.
   5.   36.  ]
 [ 1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    3.    3.    1.    2.    0.69  6.    1.
   6.   38.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.    1.    0.    0.
   0.    0.    0.    0.    0.    4.    1.    1.    1.   17.49  1.   12.
   3.   30.  ]
 [ 0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    4.    3.    0.15  1.    1.59  4.    2.
   4.   33.  ]]

Sample Label:
[0 0 0 1 0]

Class Names:
['sem desconto', 'com desconto']

Characteristics Names:
['d1', 'd2', 'd3', 'd4', 'd5', 'd7', 'd8', 'd9', 'd10', 'd11', 'd12', 'd13', 'd14', 'd15', 'd16', 

# Create Train and Test Datasets

In [3]:
# Divisão estratificada 80/20 (mantém a proporção de y=1), igual à do relatório
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"\nTrain Size: {X_train.shape}")
print(f"Test Size: {X_test.shape}")

# Valores em falta: mediana calculada só com o treino
imputer = SimpleImputer(strategy="median", add_indicator=True)
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Modelo baseado em distâncias: padronizar as variáveis (ajustado só com o treino)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


Train Size: (2940, 26)
Test Size: (736, 26)


# Train Model K-Means

In [4]:
# Há 2 classes (sem desconto / com desconto), logo n_clusters=2
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X_train)

# Cada cluster recebe a classe maioritária dos exemplos de TREINO que lhe pertencem
# (no notebook das aulas usava-se y_test, o que é fuga de informação)
train_clusters = kmeans.predict(X_train)
cluster_to_label = {c: int(mode(y_train[train_clusters == c], keepdims=True).mode[0]) for c in range(2)}
print("Mapeamento cluster -> classe:", cluster_to_label)

class Model:  # envolve o K-Means para ter .predict() devolvendo a classe
    def predict(self, X):
        return np.array([cluster_to_label[c] for c in kmeans.predict(X)])
model = Model()

Mapeamento cluster -> classe: {0: 0, 1: 0}


# Evaluate Model

In [5]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nPredictions (primeiras 30):")
print(y_pred[:30])
print("\nReal Labels (primeiras 30):")
print(y_test[:30])
print(f"\nAccuracy of the Model: {accuracy * 100:.2f}%")
print(f"Precision (classe 1): {precision:.3f} | Recall (classe 1): {recall:.3f} | F-measure: {f1:.3f}")
print("\nMatriz de confusão [[VN, FP], [FN, VP]]:")
print(confusion_matrix(y_test, y_pred))


Predictions (primeiras 30):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Real Labels (primeiras 30):
[0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0]

Accuracy of the Model: 79.48%
Precision (classe 1): 0.000 | Recall (classe 1): 0.000 | F-measure: 0.000

Matriz de confusão [[VN, FP], [FN, VP]]:
[[585   0]
 [151   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# Predict New Example

In [6]:
# Primeiro produto de task.csv (sem y conhecido)
new_product = task[features].iloc[[0]].values
new_product = imputer.transform(new_product)
new_product = scaler.transform(new_product)

predict = model.predict(new_product)
predicted_class = class_names[int(predict[0])]

print(f"New Product (i = {task['i'].iloc[0]}): {task[features].iloc[0].to_dict()}")
print(f"Predicted Class: '{predicted_class}'")

New Product (i = 1): {'d1': 0.0, 'd2': 1.0, 'd3': 0.0, 'd4': 0.0, 'd5': 0.0, 'd7': 0.0, 'd8': 0.0, 'd9': 0.0, 'd10': 0.0, 'd11': 0.0, 'd12': 0.0, 'd13': 0.0, 'd14': 0.0, 'd15': 0.0, 'd16': 0.0, 'd17': 0.0, 'd18': 0.0, 'x1': 2.0, 'x2': 2.0, 'x3': 0.5, 'x4': 1.0, 'x5': 2.69, 'x6': 8.0, 'x7': 2.0, 'x8': 3.0, 'x9': 18.0}
Predicted Class: 'sem desconto'


# Save Metrics

In [7]:
json.dump({"modelo": "K-Means", "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
            "matriz": confusion_matrix(y_test, y_pred).tolist(), "n_treino": int(len(y_train)), "n_teste": int(len(y_test))},
          open("../results/nb_kmeans.json", "w"), indent=1)